# 🏦 JPMC Consumer Banking - Hands-on Lab
## Notebook 08b: Spark on SPCS - Iceberg via Horizon REST Catalog

---

### What You'll Learn
- Running **Apache Spark** in a Snowpark Container Services (SPCS) container
- Configuring Spark to use **Snowflake Horizon REST Catalog** for Iceberg metadata
- Querying **Managed Iceberg tables WITHOUT Snowflake compute** (no warehouse used)
- Bi-directional interoperability between Spark and Snowflake

### Architecture
```
┌──────────────────────────────────┐
│   SPCS Container (Spark 3.5)     │
│   ┌────────────────────────┐     │
│   │  PySpark + Iceberg Lib │     │   ← Container compute (CPU_X64_M)
│   └──────────┬─────────────┘     │
└──────────────┼───────────────────┘
               │ REST API calls
               ▼
┌──────────────────────────────────┐
│   Snowflake Horizon REST Catalog │   ← Metadata only (no warehouse)
│   - Table discovery              │
│   - Schema information           │
│   - Data file locations          │
└──────────────┬───────────────────┘
               │ Direct file read
               ▼
┌──────────────────────────────────┐
│   Cloud Object Storage           │   ← Iceberg data files (Parquet)
│   (S3 / Azure Blob / GCS)       │
└──────────────────────────────────┘
```

> **Runtime:** Container (SPCS) - NOT a warehouse-backed notebook
> **Compute Pool:** `HOL_SPARK_POOL` (CPU_X64_M, 1-2 nodes)
> **External Access:** `HOL_MAVEN_EAI` (created in Notebook 08a)

## Step 1: Install Dependencies (Java + PySpark)

This cell installs Java (required by Spark) and PySpark. Internet access is provided by the `HOL_MAVEN_EAI` external access integration attached in Notebook 08a.

In [ ]:
import subprocess, os, glob, urllib.request, shutil
from pathlib import Path

# Verify/Install Java — PySpark 3.5 requires Java 8, 11, or 17 (NOT 21+)
compatible_javas = [c for c in glob.glob("/usr/lib/jvm/java-*-openjdk-amd64")
                    if any(f"java-{v}" in c for v in ("8", "11", "17"))]

if compatible_javas:
    os.environ["JAVA_HOME"] = sorted(compatible_javas)[-1]
    print(f"✅ JAVA_HOME = {os.environ['JAVA_HOME']}")
else:
    print("No compatible Java (8/11/17) found. Installing Java 17...")
    subprocess.run(["bash", "-c",
                    "apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless > /dev/null 2>&1"],
                   check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    print(f"✅ Java 17 installed. JAVA_HOME = {os.environ['JAVA_HOME']}")

# Install PySpark
subprocess.run(["pip", "install", "pyspark==3.5.1", "--quiet"], check=True)
print("✅ PySpark 3.5.1 installed")

# Download Iceberg JARs manually (spark.jars.packages Ivy resolver fails in SPCS)
JAR_DIR = Path("/tmp/iceberg_jars")
JAR_DIR.mkdir(exist_ok=True)

ICEBERG_JARS = {
    "iceberg-spark-runtime-3.5_2.12-1.5.0.jar":
        "https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.5.0/iceberg-spark-runtime-3.5_2.12-1.5.0.jar",
    "iceberg-aws-bundle-1.5.0.jar":
        "https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-aws-bundle/1.5.0/iceberg-aws-bundle-1.5.0.jar",
}

all_jars_ok = True
for jar_name, url in ICEBERG_JARS.items():
    jar_path = JAR_DIR / jar_name
    if jar_path.exists():
        print(f"✅ {jar_name} (cached)")
        continue
    try:
        print(f"⬇️  Downloading {jar_name}...")
        urllib.request.urlretrieve(url, str(jar_path))
        print(f"✅ {jar_name} ({jar_path.stat().st_size / 1024 / 1024:.1f} MB)")
    except Exception as e:
        print(f"❌ Failed to download {jar_name}: {e}")
        all_jars_ok = False

if all_jars_ok:
    print(f"\n✅ All Iceberg JARs ready in {JAR_DIR}")
else:
    print("\n⚠️  JAR download failed — HOL_MAVEN_EAI may not be attached to this notebook")
    print("   Fix: ALTER NOTEBOOK ... SET EXTERNAL_ACCESS_INTEGRATIONS = ('HOL_MAVEN_EAI');")

## Step 2: Configure Spark Session with Iceberg + Horizon REST Catalog

In [ ]:
# =============================================================================
# SPARK SESSION CONFIGURATION
# Configure Spark with Iceberg + Snowflake Horizon REST Catalog
# =============================================================================

from pyspark.sql import SparkSession
import os, subprocess, glob

# Ensure JAVA_HOME is set — PySpark 3.5 requires Java 8, 11, or 17 (NOT 21+)
if not os.environ.get("JAVA_HOME") or not os.path.isdir(os.environ.get("JAVA_HOME", "")):
    candidates = glob.glob("/usr/lib/jvm/java-*-openjdk-amd64")
    compatible = [c for c in candidates if any(f"java-{v}" in c for v in ("8", "11", "17"))]
    if compatible:
        os.environ["JAVA_HOME"] = sorted(compatible)[-1]
    elif candidates:
        os.environ["JAVA_HOME"] = sorted(candidates)[0]
    print(f"✅ JAVA_HOME set: {os.environ.get('JAVA_HOME')}")
else:
    print(f"✅ JAVA_HOME already set: {os.environ['JAVA_HOME']}")

# Force PATH to use Java 17 binary
java_bin_path = os.path.join(os.environ["JAVA_HOME"], "bin")
os.environ["PATH"] = f"{java_bin_path}:{os.environ['PATH']}"

# Set SPARK_HOME
import pyspark
os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)
print(f"✅ JAVA_HOME = {os.environ['JAVA_HOME']}")
print(f"✅ SPARK_HOME = {os.environ['SPARK_HOME']}")

# Get Snowflake connection details
from snowflake.snowpark.context import get_active_session
_sf_session = get_active_session()
SNOWFLAKE_ACCOUNT = _sf_session.sql("SELECT CURRENT_ACCOUNT()").collect()[0][0]

# Get the org-account identifier (required for Horizon REST endpoint)
_org = _sf_session.sql("SELECT CURRENT_ORGANIZATION_NAME()").collect()[0][0]
_acct = _sf_session.sql("SELECT CURRENT_ACCOUNT_NAME()").collect()[0][0]
ACCOUNT_IDENTIFIER = f"{_org}-{_acct}"
SNOWFLAKE_HOST = os.environ.get("SNOWFLAKE_HOST", f"{ACCOUNT_IDENTIFIER}.snowflakecomputing.com")

# Get OAuth token from SPCS session token file
TOKEN_FILE = "/snowflake/session/token"
if os.path.exists(TOKEN_FILE):
    with open(TOKEN_FILE, "r") as f:
        SPCS_TOKEN = f.read().strip()
    print(f"✅ SPCS session token loaded ({len(SPCS_TOKEN)} chars)")
else:
    SPCS_TOKEN = os.environ.get("SNOWFLAKE_TOKEN", "")
    if SPCS_TOKEN:
        print(f"✅ Token from env var ({len(SPCS_TOKEN)} chars)")
    else:
        print("⚠️  No token found!")

# --- PAT for Horizon REST Catalog ---
# The SPCS session token is NOT valid for the Horizon REST API.
# Read PAT from the table created in 08a_Spark_SPCS_Prep.
_pat_row = _sf_session.sql(
    "SELECT * FROM JPMC_CONSUMER_BANKING_HOL.HOL_LAB.HOL_PAT_STORE LIMIT 1"
).collect()[0]
HORIZON_TOKEN = _pat_row[1]  # second column is the token value
print(f"✅ Horizon PAT loaded from HOL_PAT_STORE ({len(HORIZON_TOKEN)} chars)")

# Horizon REST Catalog endpoint
CATALOG_URI = f"https://{SNOWFLAKE_HOST}/polaris/api/catalog"
print(f"   Catalog URI: {CATALOG_URI}")
print(f"   Account: {ACCOUNT_IDENTIFIER}")

# Ensure no stale SparkContext is running (would prevent JAR loading)
try:
    from pyspark import SparkContext
    sc = SparkContext._active_spark_context
    if sc is not None:
        print("⚠️  Stopping stale SparkContext to allow fresh JAR loading...")
        sc.stop()
except Exception:
    pass

# Build Spark session with Iceberg extensions
# Using spark.jars (pre-downloaded) instead of spark.jars.packages (Ivy resolver)
iceberg_jar = "/tmp/iceberg_jars/iceberg-spark-runtime-3.5_2.12-1.5.0.jar"
aws_bundle_jar = "/tmp/iceberg_jars/iceberg-aws-bundle-1.5.0.jar"
for jar in [iceberg_jar, aws_bundle_jar]:
    if not os.path.exists(jar):
        raise FileNotFoundError(f"JAR not found: {jar} — re-run cell 2")

all_jars = f"{iceberg_jar},{aws_bundle_jar}"
print(f"🔄 Starting Spark with JARs: {all_jars}")

if not HORIZON_TOKEN:
    raise RuntimeError("No Horizon PAT available — see setup instructions above")

# Use PAT with credential property + oauth2-server-uri for token exchange
# The Iceberg REST client exchanges the PAT for an access token via OAuth2 client_credentials.
# For Horizon Catalog, the PAT goes in 'credential' field.
# Iceberg 1.5.0 expects credential format "client_id:client_secret" — for Snowflake PAT,
# use the PAT as client_secret with no client_id (empty string before colon).
builder = SparkSession.builder \
    .master("local[*]") \
    .appName("JPMC_HOL_Iceberg_Interop") \
    .config("spark.jars", all_jars) \
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.snowflake_catalog",
            "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.snowflake_catalog.type", "rest") \
    .config("spark.sql.catalog.snowflake_catalog.uri", CATALOG_URI) \
    .config("spark.sql.catalog.snowflake_catalog.warehouse",
            "JPMC_CONSUMER_BANKING_HOL") \
    .config("spark.sql.catalog.snowflake_catalog.credential", HORIZON_TOKEN) \
    .config("spark.sql.catalog.snowflake_catalog.scope",
            "session:role:ACCOUNTADMIN") \
    .config("spark.sql.catalog.snowflake_catalog.header.X-Iceberg-Access-Delegation",
            "vended-credentials") \
    .config("spark.sql.catalog.snowflake_catalog.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.snowflake_catalog.client.region", "us-west-2") \
    .config("spark.sql.iceberg.vectorization.enabled", "false") \
    .config("spark.sql.defaultCatalog", "snowflake_catalog")

spark = builder.getOrCreate()

print(f"\n✅ Spark session created: {spark.version}")
print(f"   Catalog URI: {CATALOG_URI}")
print(f"   Database: JPMC_CONSUMER_BANKING_HOL")

## Step 3: Discover Tables via Horizon Catalog

In [ ]:
# =============================================================================
# DISCOVER TABLES via Horizon REST Catalog
# No Snowflake warehouse is used for any of these operations!
# =============================================================================

# List available namespaces (schemas)
spark.sql("SHOW NAMESPACES IN snowflake_catalog").show()

# List tables in HOL_LAB schema
spark.sql("SHOW TABLES IN snowflake_catalog.HOL_LAB").show()

# Note: Only Iceberg tables are visible through the REST catalog
# Native Snowflake tables (CUSTOMERS, TRANSACTIONS, SUPPORT_TICKETS) 
# are NOT accessible via this path - by design!

## Step 4: Read Iceberg Tables with Spark (No Snowflake Warehouse!)

In [ ]:
# =============================================================================
# READ ICEBERG TABLES DIRECTLY WITH SPARK
# Zero Snowflake warehouse credits consumed!
# =============================================================================

# Read ACCOUNTS table (managed Iceberg)
accounts_df = spark.read.table("snowflake_catalog.HOL_LAB.ACCOUNTS")
print(f"ACCOUNTS: {accounts_df.count()} rows, {len(accounts_df.columns)} columns")
accounts_df.printSchema()
accounts_df.show(5)

In [ ]:
# Read LOANS table
loans_df = spark.read.table("snowflake_catalog.HOL_LAB.LOANS")
print(f"LOANS: {loans_df.count()} rows")

# Read CREDIT_CARDS table
credit_cards_df = spark.read.table("snowflake_catalog.HOL_LAB.CREDIT_CARDS")
print(f"CREDIT_CARDS: {credit_cards_df.count()} rows")

## Step 5: Run Spark Transformations on Iceberg Data

In [ ]:
# =============================================================================
# SPARK ANALYTICS ON ICEBERG TABLES
# Same data that Snowflake DTs process, now processed by Spark!
# =============================================================================

from pyspark.sql import functions as F

# Analysis 1: Portfolio summary across all three Iceberg tables
portfolio_summary = accounts_df.join(loans_df, "CUSTOMER_ID", "left") \
    .join(credit_cards_df, "CUSTOMER_ID", "left") \
    .groupBy("ACCOUNT_TYPE") \
    .agg(
        F.count("*").alias("total_records"),
        F.sum("BALANCE").alias("total_account_balance"),
        F.sum(loans_df["CURRENT_BALANCE"]).alias("total_loan_balance"),
        F.sum(credit_cards_df["CURRENT_BALANCE"]).alias("total_cc_balance"),
        F.avg(accounts_df["INTEREST_RATE"]).alias("avg_account_rate")
    )

print("Portfolio Summary (computed by Spark, data from Snowflake Iceberg):")
portfolio_summary.show()

In [ ]:
# Analysis 2: Loan risk distribution (same as Snowflake DT LOAN_RISK_PROFILE)
loan_risk_spark = loans_df.withColumn(
    "risk_category",
    F.when(F.col("STATUS").isin("DEFAULT"), "CRITICAL")
     .when(F.col("STATUS").isin("DELINQUENT_90"), "CRITICAL")
     .when(F.col("STATUS").isin("DELINQUENT_60", "DELINQUENT_30"), "HIGH")
     .when(F.col("DEBT_TO_INCOME_RATIO") > 0.45, "MEDIUM")
     .otherwise("LOW")
)

print("Loan Risk Distribution (Spark-computed):")
loan_risk_spark.groupBy("risk_category") \
    .agg(
        F.count("*").alias("loan_count"),
        F.sum("CURRENT_BALANCE").alias("total_exposure"),
        F.avg("INTEREST_RATE").alias("avg_rate")
    ) \
    .orderBy("total_exposure", ascending=False) \
    .show()

In [ ]:
# Analysis 3: Use SparkSQL for familiar SQL syntax on Iceberg tables
spark.sql("""
    SELECT 
        LOAN_TYPE,
        STATUS,
        COUNT(*) as loan_count,
        ROUND(AVG(PRINCIPAL_AMOUNT), 2) as avg_principal,
        ROUND(AVG(CURRENT_BALANCE), 2) as avg_balance,
        ROUND(AVG(INTEREST_RATE), 3) as avg_rate
    FROM snowflake_catalog.HOL_LAB.LOANS
    GROUP BY LOAN_TYPE, STATUS
    HAVING COUNT(*) > 100
    ORDER BY LOAN_TYPE, loan_count DESC
""").show(20)

## Step 8: Compare - Same Analysis, Different Engines

| Metric | Snowflake DT (NB 03) | Spark on SPCS (NB 08b) |
|--------|----------------------|----------------------|
| **Data Format** | Read from Iceberg | Read from Iceberg |
| **Compute** | Snowflake Warehouse | SPCS Container (Spark) |
| **Catalog** | Snowflake Internal | Horizon REST Catalog |
| **Result** | Same analytical output | Same analytical output |
| **Cost Model** | Credits/second (warehouse) | Credits/hour (container) |
| **Best For** | SQL-centric teams | Existing Spark pipelines |
| **Latency** | Sub-second for cached | Startup time + processing |

### Key Takeaway
With Managed Iceberg Tables + Horizon Catalog:
- **One copy of data** serves multiple engines
- **No data movement** between Snowflake and Spark
- **Choose the right engine** for each workload without data silos
- **Governance stays unified** through Snowflake's access control

In [ ]:
# Cleanup: Stop Spark session
spark.stop()
print("✅ Spark session stopped")